# VisoLabel Colab Trainer

Use this notebook when VisoLabel gives you a Colab training token.

1. In VisoLabel, choose `Train on Colab` and copy the token.
2. In Colab, set `Runtime > Change runtime type > GPU`.
3. Run the form cell below.
4. Paste the token and click `Run full pipeline`.

The notebook downloads the encrypted bundle, decrypts it locally in Colab, validates the dataset, trains the selected model, and uploads checkpoints back to VisoLabel. **Training progress is shown below — keep this tab open while training runs.**


In [ ]:
#@title VisoLabel Colab Trainer { display-mode: "form" }
#@markdown Paste the token from VisoLabel and click **Run full pipeline**. Training progress will be displayed in the cell below as the model trains.
import base64
import hashlib
import html as _html
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
import threading
import zipfile
from pathlib import Path

os.environ.setdefault("PYTHONUNBUFFERED", "1")
os.environ.setdefault("WANDB_DISABLED", "true")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

def _pip_install_once():
    import importlib.util
    required = {
        "requests": "requests",
        "cryptography": "cryptography",
        "ipywidgets": "ipywidgets",
        "tqdm": "tqdm",
    }
    missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

_pip_install_once()

import requests
from IPython.display import HTML, display
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
from tqdm.auto import tqdm

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass

import ipywidgets as widgets

DEFAULT_API_BASE_URL = os.environ.get("VISOLABEL_API_BASE", "https://api.visolabel.ovh")
TOKEN_PREFIX = "VL1."
LEGACY_TOKEN_PREFIX = "VL-COLAB-ENC1."
WORK_DIR = Path("/content/visolabel_run")
BUNDLE_ENCRYPTED = WORK_DIR / "bundle.zip.vlenc"
BUNDLE_ZIP = WORK_DIR / "bundle.zip"
BUNDLE_DIR = WORK_DIR / "bundle"
OUTPUT_DIR = WORK_DIR / "output"
RUNNER_PATH = WORK_DIR / "visolabel_train_runner.py"
TRAINING_LOG = OUTPUT_DIR / "training.log"
METRICS_PATH = OUTPUT_DIR / "metrics.json"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
ARTIFACT_EXTENSIONS = {".pth", ".pt", ".ckpt", ".onnx", ".log", ".json", ".yaml", ".yml", ".txt", ".csv"}


RUNNER_CODE = r"""
import json
import math
import os
import shutil
import sys
import time
from pathlib import Path

os.environ.setdefault("PYTHONUNBUFFERED", "1")
os.environ.setdefault("WANDB_DISABLED", "true")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")


def log(message):
    print(f"[visolabel-train] {message}", flush=True)


def import_attr(module, names):
    for name in names:
        if hasattr(module, name):
            return getattr(module, name), name
    raise RuntimeError("None of these RF-DETR classes are available: " + ", ".join(names))


def model_class_for_name(model_name):
    import rfdetr
    name = str(model_name or "RF-DETR-N")
    upper = name.upper()
    is_seg = "SEG" in upper
    suffix = upper.split("-")[-1]
    if suffix == "2XL":
        det = ["RFDETR2XLarge", "RFDETRXLarge", "RFDETRLarge"]
        seg = ["RFDETRSeg2XLarge", "RFDETRSegXLarge", "RFDETRSegLarge"]
    elif suffix in {"XL", "X"}:
        det = ["RFDETRXLarge", "RFDETRLarge"]
        seg = ["RFDETRSegXLarge", "RFDETRSegLarge"]
    elif suffix in {"L", "LARGE"}:
        det = ["RFDETRLarge", "RFDETRBase"]
        seg = ["RFDETRSegLarge", "RFDETRSegMedium", "RFDETRSegSmall", "RFDETRSegNano"]
    elif suffix in {"M", "MEDIUM"}:
        det = ["RFDETRMedium", "RFDETRBase"]
        seg = ["RFDETRSegMedium", "RFDETRSegSmall", "RFDETRSegNano"]
    elif suffix in {"S", "SMALL"}:
        det = ["RFDETRSmall", "RFDETRBase"]
        seg = ["RFDETRSegSmall", "RFDETRSegNano"]
    else:
        det = ["RFDETRNano", "RFDETRBase"]
        seg = ["RFDETRSegNano", "RFDETRSegSmall"]
    return import_attr(rfdetr, seg if is_seg else det)


def nearest_valid_resolution(requested, block_size):
    requested = int(requested or 560)
    block_size = max(int(block_size or 1), 1)
    if requested % block_size == 0:
        return requested
    lower = max(block_size, (requested // block_size) * block_size)
    upper = lower + block_size
    return lower if requested - lower <= upper - requested else upper


def apply_resolution(model, requested):
    config = getattr(model, "model_config", None)
    if config is None:
        return int(requested or 560)
    patch_size = int(getattr(config, "patch_size", 14) or 14)
    num_windows = int(getattr(config, "num_windows", 1) or 1)
    block_size = patch_size * num_windows
    resolution = nearest_valid_resolution(requested, block_size)
    current_resolution = int(getattr(config, "resolution", resolution) or resolution)
    current_pe = getattr(config, "positional_encoding_size", None)
    if current_pe is not None:
        derived_pe = current_resolution // patch_size
        if int(current_pe) == int(derived_pe):
            setattr(config, "positional_encoding_size", resolution // patch_size)
    setattr(config, "resolution", resolution)
    wrapped = getattr(model, "model", None)
    if wrapped is not None:
        if hasattr(wrapped, "resolution"):
            wrapped.resolution = resolution
        args = getattr(wrapped, "args", None)
        if args is not None:
            if hasattr(args, "resolution"):
                args.resolution = resolution
            if hasattr(args, "positional_encoding_size") and current_pe is not None and int(current_pe) == int(current_resolution // patch_size):
                args.positional_encoding_size = resolution // patch_size
    if resolution != int(requested or resolution):
        log(f"Adjusted RF-DETR resolution from {requested} to {resolution}; model requires multiples of {block_size}.")
    return resolution


def make_train_kwargs(cfg, dataset_dir, output_dir, resolution):
    batch_size = int(cfg.get("batch_size", 4) or 4)
    grad_accum_steps = int(cfg.get("grad_accum_steps") or max(1, math.ceil(16 / max(batch_size, 1))))
    kwargs = {
        "dataset_dir": str(dataset_dir),
        "output_dir": str(output_dir),
        "epochs": int(cfg.get("epochs", 50) or 50),
        "batch_size": batch_size,
        "grad_accum_steps": grad_accum_steps,
        "lr": float(cfg.get("learning_rate", cfg.get("lr", 1e-4)) or 1e-4),
        "resolution": int(resolution),
        "class_names": cfg.get("classes") or None,
        "device": "cuda",
        "num_workers": int(cfg.get("num_workers", 2) or 2),
        "progress_bar": True,
        "tensorboard": False,
        "wandb": False,
        "mlflow": False,
        "clearml": False,
        "checkpoint_interval": int(cfg.get("checkpoint_interval", 1) or 1),
        "eval_interval": int(cfg.get("eval_interval", 1) or 1),
        "eval_max_dets": int(cfg.get("eval_max_dets", 500) or 500),
        "log_per_class_metrics": True,
        "compute_val_loss": False,
        "run_test": False,
    }
    return {k: v for k, v in kwargs.items() if v is not None}


def build_train_config(model, kwargs):
    attempt = dict(kwargs)
    attempt.pop("resolution", None)
    attempt.pop("device", None)
    optional_order = [
        "run_test", "compute_val_loss", "log_per_class_metrics", "eval_interval",
        "eval_max_dets", "clearml", "mlflow", "class_names", "num_workers",
    ]
    while True:
        try:
            return model.get_train_config(**attempt), attempt
        except Exception as exc:
            removed = None
            for key in optional_order:
                if key in attempt:
                    removed = key
                    attempt.pop(key, None)
                    log(f"RF-DETR TrainConfig rejected '{key}', retrying without it.")
                    break
            if removed is None:
                raise exc



def _json_metric_value(value):
    try:
        if hasattr(value, "detach"):
            value = value.detach()
        if hasattr(value, "cpu"):
            value = value.cpu()
        if hasattr(value, "item"):
            return float(value.item())
        if hasattr(value, "tolist"):
            return value.tolist()
    except Exception:
        pass
    if isinstance(value, (int, float, str, bool)) or value is None:
        return value
    if isinstance(value, (list, tuple)):
        return [_json_metric_value(v) for v in value]
    if isinstance(value, dict):
        return {str(k): _json_metric_value(v) for k, v in value.items()}
    try:
        return float(value)
    except Exception:
        return str(value)


def _write_live_metrics(output_dir, latest, history=None):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    latest = {str(k): _json_metric_value(v) for k, v in dict(latest or {}).items()}
    latest["updated_at"] = time.time()
    rows = list(history or [])
    if not rows:
        metrics_path = output_dir / "metrics.json"
        if metrics_path.exists():
            try:
                previous = json.loads(metrics_path.read_text(encoding="utf-8"))
                if isinstance(previous, dict) and isinstance(previous.get("history"), list):
                    rows = previous["history"]
            except Exception:
                rows = []
    rows.append(latest)
    payload = {"latest": latest, "history": rows[-500:]}
    (output_dir / "metrics.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")


class LiveProgressCallback:
    # Writes a small progress.json file every batch (rate-limited). The
    # notebook GUI polls this file to drive the live training progress card.

    def __init__(self, output_dir, total_epochs):
        self.output_dir = Path(output_dir)
        self.total_epochs = int(total_epochs or 0)
        self.last_write = 0.0
        self.start_time = time.time()
        self.last_batch_count = 0
        self.last_batch_time = self.start_time
        self.throughput_ema = None
        self._batch_size_hint = 0

    def _resolve_loss(self, trainer, outputs):
        for key in ("loss", "train_loss", "loss_total", "loss_ce"):
            value = getattr(trainer, "callback_metrics", {}).get(key)
            if value is None:
                continue
            try:
                return float(value.item() if hasattr(value, "item") else value)
            except Exception:
                continue
        if isinstance(outputs, dict):
            loss = outputs.get("loss")
            if loss is not None:
                try:
                    return float(loss.item() if hasattr(loss, "item") else loss)
                except Exception:
                    return None
        return None

    def _write(self, payload):
        try:
            self.output_dir.mkdir(parents=True, exist_ok=True)
            (self.output_dir / "progress.json").write_text(
                json.dumps(payload), encoding="utf-8"
            )
        except Exception:
            pass

    def _maybe_write(self, payload, force=False):
        now = time.time()
        if not force and now - self.last_write < 0.7:
            return
        self.last_write = now
        payload["updated_at"] = now
        self._write(payload)

    def on_train_start(self, trainer, pl_module):
        total_epochs = int(getattr(trainer, "max_epochs", 0) or self.total_epochs or 0)
        if total_epochs:
            self.total_epochs = total_epochs
        self._maybe_write(
            {
                "current_epoch": 1,
                "total_epochs": self.total_epochs,
                "current_iteration": 0,
                "total_iterations": 0,
                "loss": None,
                "throughput": None,
                "elapsed": 0.0,
                "status": "starting",
            },
            force=True,
        )

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        total_epochs = int(getattr(trainer, "max_epochs", 0) or self.total_epochs or 0)
        if total_epochs:
            self.total_epochs = total_epochs
        total_iter = int(getattr(trainer, "num_training_batches", 0) or 0)
        current_iter = int(batch_idx) + 1
        epoch = int(getattr(trainer, "current_epoch", 0) or 0) + 1

        now = time.time()
        batch_size = 0
        try:
            if isinstance(batch, (list, tuple)) and batch:
                first = batch[0]
                if hasattr(first, "shape"):
                    batch_size = int(first.shape[0])
                elif isinstance(first, (list, tuple)):
                    batch_size = len(first)
        except Exception:
            batch_size = 0
        if batch_size:
            self._batch_size_hint = batch_size

        dt = max(now - self.last_batch_time, 1e-6)
        instant_throughput = (self._batch_size_hint or 1) / dt
        if self.throughput_ema is None:
            self.throughput_ema = instant_throughput
        else:
            self.throughput_ema = 0.7 * self.throughput_ema + 0.3 * instant_throughput
        self.last_batch_time = now

        loss = self._resolve_loss(trainer, outputs)
        self._maybe_write(
            {
                "current_epoch": epoch,
                "total_epochs": self.total_epochs,
                "current_iteration": current_iter,
                "total_iterations": total_iter,
                "loss": loss,
                "throughput": self.throughput_ema,
                "elapsed": now - self.start_time,
                "status": "training",
            }
        )

    def on_train_epoch_end(self, trainer, pl_module):
        epoch = int(getattr(trainer, "current_epoch", 0) or 0) + 1
        self._maybe_write(
            {
                "current_epoch": epoch,
                "total_epochs": self.total_epochs,
                "current_iteration": int(getattr(trainer, "num_training_batches", 0) or 0),
                "total_iterations": int(getattr(trainer, "num_training_batches", 0) or 0),
                "loss": self._resolve_loss(trainer, None),
                "throughput": self.throughput_ema,
                "elapsed": time.time() - self.start_time,
                "status": "epoch_end",
            },
            force=True,
        )


class LiveMetricsCallback:
    def __init__(self, output_dir, total_epochs):
        self.output_dir = Path(output_dir)
        self.total_epochs = int(total_epochs or 0)
        self.history = []

    def capture(self, trainer):
        metrics = {}
        for key, value in getattr(trainer, "callback_metrics", {}).items():
            metrics[str(key)] = _json_metric_value(value)
        epoch = int(getattr(trainer, "current_epoch", 0) or 0) + 1
        metrics.setdefault("epoch", epoch)
        if self.total_epochs:
            metrics.setdefault("total_epochs", self.total_epochs)
        if metrics:
            self.history.append(metrics)
            _write_live_metrics(self.output_dir, metrics, self.history)

    def on_train_epoch_end(self, trainer, pl_module):
        self.capture(trainer)

    def on_validation_epoch_end(self, trainer, pl_module):
        self.capture(trainer)

def train_with_custom_lightning(model, kwargs):
    from rfdetr.training import RFDETRDataModule, RFDETRModelModule, build_trainer
    train_config, used_kwargs = build_train_config(model, kwargs)
    dataset_dir = getattr(train_config, "dataset_dir", None)
    if dataset_dir and hasattr(model, "_align_num_classes_from_dataset"):
        model._align_num_classes_from_dataset(dataset_dir)
    module = RFDETRModelModule(model.model_config, train_config)
    datamodule = RFDETRDataModule(model.model_config, train_config)
    trainer = build_trainer(
        train_config,
        model.model_config,
        accelerator="gpu" if os.environ.get("CUDA_VISIBLE_DEVICES", "") != "-1" else "auto",
        devices=1,
        num_sanity_val_steps=0,
        log_every_n_steps=1,
        enable_progress_bar=True,
    )
    output_dir_for_callbacks = kwargs.get("output_dir") or getattr(train_config, "output_dir", ".")
    total_epochs_for_callbacks = kwargs.get("epochs") or getattr(train_config, "epochs", 0)
    live_metrics = LiveMetricsCallback(output_dir_for_callbacks, total_epochs_for_callbacks)
    live_progress = LiveProgressCallback(output_dir_for_callbacks, total_epochs_for_callbacks)
    try:
        trainer.callbacks.append(live_metrics)
        trainer.callbacks.append(live_progress)
    except Exception as exc:
        log(f"Could not attach live metrics callback: {exc}")
    log("Starting RF-DETR Lightning fit with sanity validation disabled.")
    trainer.fit(module, datamodule, ckpt_path=getattr(train_config, "resume", None) or None)
    live_metrics.capture(trainer)
    if hasattr(model, "model") and hasattr(model.model, "model"):
        model.model.model = module.model
    return used_kwargs


def train_with_high_level(model, kwargs):
    log("Starting RF-DETR high-level train() fallback.")
    return model.train(**kwargs)


def run_detection(manifest_path, dataset_dir, output_dir):
    manifest = json.loads(Path(manifest_path).read_text(encoding="utf-8"))
    cfg = manifest.get("training_config", {})
    model_name = cfg.get("model_name", "RF-DETR-N")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    cls, selected_name = model_class_for_name(model_name)
    log(f"Requested model {model_name}; using {selected_name}.")
    model = cls()
    resolution = apply_resolution(model, cfg.get("image_size", cfg.get("resolution", 560)))
    kwargs = make_train_kwargs(cfg, dataset_dir, output_dir, resolution)
    try:
        train_with_custom_lightning(model, kwargs)
    except Exception as exc:
        log(f"Custom Lightning path failed: {exc}")
        train_with_high_level(model, kwargs)
    log("RF-DETR training finished.")


def run_classifier(manifest_path, dataset_dir, output_dir):
    import torch
    from torch import nn
    from torch.utils.data import DataLoader
    from torchvision import datasets, transforms
    import timm

    manifest = json.loads(Path(manifest_path).read_text(encoding="utf-8"))
    cfg = manifest.get("training_config", {})
    image_size = int(cfg.get("image_size", 224) or 224)
    batch_size = int(cfg.get("batch_size", 16) or 16)
    epochs = int(cfg.get("epochs", 30) or 30)
    lr = float(cfg.get("learning_rate", 3e-4) or 3e-4)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    train_dir = Path(dataset_dir) / "train"
    valid_dir = Path(dataset_dir) / "valid"
    if not valid_dir.exists():
        valid_dir = Path(dataset_dir) / "val"
    if not train_dir.exists():
        raise RuntimeError(f"Classification train directory missing: {train_dir}")
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(image_size),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    val_tf = transforms.Compose([
        transforms.Resize(int(image_size * 1.14)),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    train_ds = datasets.ImageFolder(str(train_dir), transform=train_tf)
    val_ds = datasets.ImageFolder(str(valid_dir), transform=val_tf) if valid_dir.exists() else None
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True) if val_ds else None
    model_name = str(cfg.get("model_name", "convnext_tiny.fb_in22k_ft_in1k"))
    model = timm.create_model(model_name, pretrained=True, num_classes=len(train_ds.classes))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=float(cfg.get("label_smoothing", 0.0) or 0.0))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=float(cfg.get("weight_decay", 0.05) or 0.05))
    best_path = output_dir / "checkpoint_best_total.pth"
    best_acc = -1.0
    metrics_history = []
    log(f"Starting {model_name} classifier training on {device}.")
    progress_path = output_dir / "progress.json"
    train_start = time.time()
    last_progress_write = 0.0

    def _write_progress(current_iter, total_iter, current_epoch, loss_value, status="training"):
        nonlocal last_progress_write
        now = time.time()
        if status == "training" and now - last_progress_write < 0.5:
            return
        last_progress_write = now
        try:
            progress_path.write_text(json.dumps({
                "current_epoch": current_epoch,
                "total_epochs": epochs,
                "current_iteration": current_iter,
                "total_iterations": total_iter,
                "loss": float(loss_value) if loss_value is not None else None,
                "throughput": (current_iter * batch_size) / max(now - train_start, 1e-6) if current_iter else None,
                "elapsed": now - train_start,
                "status": status,
                "updated_at": now,
            }), encoding="utf-8")
        except Exception:
            pass

    for epoch in range(epochs):
        epoch_started = time.time()
        model.train()
        total_loss = 0.0
        total_batches = len(train_loader)
        _write_progress(0, total_batches, epoch + 1, None, status="epoch_start")
        for batch_idx, (x, y) in enumerate(train_loader):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(x), y)
            loss.backward()
            optimizer.step()
            total_loss += float(loss.item())
            _write_progress(batch_idx + 1, total_batches, epoch + 1, loss.item())
        avg_loss = total_loss / max(len(train_loader), 1)
        _write_progress(total_batches, total_batches, epoch + 1, avg_loss, status="epoch_end")
        epoch_seconds = max(time.time() - epoch_started, 1e-6)
        throughput = len(train_ds) / epoch_seconds
        if val_loader:
            model.eval()
            correct = 0
            total = 0
            with torch.no_grad():
                for x, y in val_loader:
                    x, y = x.to(device), y.to(device)
                    pred = model(x).argmax(dim=1)
                    correct += int((pred == y).sum().item())
                    total += int(y.numel())
            acc = 100.0 * correct / max(total, 1)
            log(f"Epoch {epoch + 1}/{epochs}: loss={avg_loss:.4f}, val_acc={acc:.2f}%")
            metrics = {"epoch": epoch + 1, "total_epochs": epochs, "loss": avg_loss, "accuracy": acc, "val_accuracy": acc, "throughput": throughput}
            metrics_history.append(metrics)
            _write_live_metrics(output_dir, metrics, metrics_history)
            checkpoint_path = output_dir / "checkpoint.pth"
            torch.save({"model_state_dict": model.state_dict(), "classes": train_ds.classes, "model_name": model_name, "epoch": epoch + 1}, checkpoint_path)
            if acc >= best_acc:
                best_acc = acc
                torch.save({"model_state_dict": model.state_dict(), "classes": train_ds.classes, "model_name": model_name, "epoch": epoch + 1}, best_path)
        else:
            checkpoint_path = output_dir / "checkpoint.pth"
            torch.save({"model_state_dict": model.state_dict(), "classes": train_ds.classes, "model_name": model_name, "epoch": epoch + 1}, checkpoint_path)
            log(f"Epoch {epoch + 1}/{epochs}: loss={avg_loss:.4f}")
            metrics = {"epoch": epoch + 1, "total_epochs": epochs, "loss": avg_loss, "throughput": throughput}
            metrics_history.append(metrics)
            _write_live_metrics(output_dir, metrics, metrics_history)
    if not best_path.exists():
        torch.save({"model_state_dict": model.state_dict(), "classes": train_ds.classes, "model_name": model_name}, best_path)
    log("Classifier training finished.")


def main():
    manifest_path = Path(sys.argv[1])
    dataset_dir = Path(sys.argv[2])
    output_dir = Path(sys.argv[3])
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    command = manifest.get("task", {}).get("command", "train")
    if command == "classify":
        run_classifier(manifest_path, dataset_dir, output_dir)
    elif command == "train":
        run_detection(manifest_path, dataset_dir, output_dir)
    else:
        raise RuntimeError(f"Unsupported training command: {command}")


if __name__ == "__main__":
    main()
"""


def _b64url_decode(value: str) -> bytes:
    value = value.strip()
    return base64.urlsafe_b64decode(value + "=" * (-len(value) % 4))


def _normalize_url(base: str, endpoint: str) -> str:
    return base.rstrip("/") + "/" + endpoint.lstrip("/")


def _format_bytes(size: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    value = float(size)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f"{value:.1f} {unit}" if unit != "B" else f"{int(value)} B"
        value /= 1024


def _sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def _count_images(path: Path) -> int:
    if not path or not path.exists():
        return 0
    return sum(1 for p in path.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)


def _safe_extract_zip(zip_path: Path, target: Path):
    target_resolved = target.resolve()
    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in zf.infolist():
            destination = (target / member.filename).resolve()
            if not str(destination).startswith(str(target_resolved)):
                raise RuntimeError(f"Unsafe zip path: {member.filename}")
        zf.extractall(target)


def _normalize_encryption_metadata(metadata: dict) -> dict:
    if not isinstance(metadata, dict):
        return {}
    if metadata.get("salt_b64") and metadata.get("nonce_b64") and metadata.get("tag_b64"):
        normalized = dict(metadata)
    else:
        normalized = {
            "enabled": True,
            "algorithm": metadata.get("algorithm", "AES-256-GCM"),
            "kdf": metadata.get("kdf", "PBKDF2-HMAC-SHA256"),
            "kdf_iterations": metadata.get("kdf_iterations", metadata.get("i", 200000)),
            "salt_b64": metadata.get("s") or metadata.get("salt_b64") or metadata.get("salt", ""),
            "nonce_b64": metadata.get("n") or metadata.get("nonce_b64") or metadata.get("nonce", ""),
            "tag_b64": metadata.get("g") or metadata.get("tag_b64") or metadata.get("tag") or metadata.get("t", ""),
            "aad_b64": metadata.get("a") or metadata.get("aad_b64") or metadata.get("aad", ""),
            "plaintext_sha256": metadata.get("ps") or metadata.get("plaintext_sha256") or metadata.get("psha256", ""),
            "ciphertext_sha256": metadata.get("cs") or metadata.get("ciphertext_sha256") or metadata.get("csha256", ""),
            "plaintext_size": metadata.get("psz") or metadata.get("plaintext_size", 0),
            "ciphertext_size": metadata.get("csz") or metadata.get("ciphertext_size", 0),
        }
    normalized.setdefault("algorithm", "AES-256-GCM")
    normalized.setdefault("kdf", "PBKDF2-HMAC-SHA256")
    normalized.setdefault("kdf_iterations", normalized.get("iterations", 200000))
    normalized.setdefault("aad_b64", "dmlzb2xhYmVsLWNvbGFiLWJ1bmRsZS12MQ==")
    if not normalized.get("salt_b64") or not normalized.get("nonce_b64") or not normalized.get("tag_b64"):
        return {}
    return normalized


def _extract_bundle_url(value: dict) -> str:
    if not isinstance(value, dict):
        return ""
    for key in ("bundle_url", "download_url", "presigned_download_url", "s3_url", "url"):
        found = str(value.get(key) or "")
        if found:
            return found
    nested = value.get("bundle") or value.get("data") or {}
    if isinstance(nested, dict):
        return _extract_bundle_url(nested)
    return ""


def _extract_encryption_metadata(value: dict) -> dict:
    if not isinstance(value, dict):
        return {}
    direct = _normalize_encryption_metadata(value.get("encryption") or value.get("encryption_metadata") or value.get("e") or {})
    if direct:
        return direct
    nested = value.get("bundle") or value.get("data") or {}
    if isinstance(nested, dict):
        return _extract_encryption_metadata(nested)
    return {}


def _parse_encrypted_token(token: str):
    prefix = TOKEN_PREFIX if token.startswith(TOKEN_PREFIX) else LEGACY_TOKEN_PREFIX
    payload = json.loads(_b64url_decode(token[len(prefix):]).decode("utf-8"))
    server_token = ""
    password = ""
    api_url = ""
    encryption = {}
    bundle_url = ""
    bundle_id = ""
    if isinstance(payload, list):
        if len(payload) >= 1:
            server_token = str(payload[0] or "")
        if len(payload) >= 2:
            password = str(payload[1] or "")
        if len(payload) >= 3:
            if isinstance(payload[2], dict):
                encryption = _normalize_encryption_metadata(payload[2])
            else:
                api_url = str(payload[2] or "")
        if len(payload) >= 4 and isinstance(payload[3], dict):
            encryption = _normalize_encryption_metadata(payload[3])
    elif isinstance(payload, dict):
        server_token = str(payload.get("server_token") or payload.get("token") or "")
        password = str(payload.get("password") or "")
        api_url = str(payload.get("api_url") or payload.get("base_url") or "")
        bundle_url = _extract_bundle_url(payload)
        encryption = _extract_encryption_metadata(payload)
        bundle_id = str(payload.get("bundle_id") or "")
    else:
        raise RuntimeError("Encrypted token payload is not valid JSON.")
    return {
        "server_token": server_token,
        "password": password,
        "api_url": api_url,
        "bundle_url": bundle_url,
        "encryption": encryption,
        "bundle_id": bundle_id,
    }


def _validate_coco_split(split_dir: Path):
    ann_path = split_dir / "_annotations.coco.json"
    result = {"split": split_dir.name, "images": 0, "annotations": 0, "categories": 0, "removed_annotations": 0}
    if not ann_path.exists():
        if split_dir.name == "valid":
            alt = split_dir.parent / "val" / "_annotations.coco.json"
            if alt.exists():
                shutil.copytree(alt.parent, split_dir, dirs_exist_ok=True)
            else:
                raise RuntimeError(f"Missing COCO annotations: {ann_path}")
        else:
            raise RuntimeError(f"Missing COCO annotations: {ann_path}")
    data = json.loads(ann_path.read_text(encoding="utf-8"))
    images = data.get("images") or []
    anns = data.get("annotations") or []
    categories = data.get("categories") or []
    image_ids = {img.get("id") for img in images}
    category_ids = {cat.get("id") for cat in categories}
    valid_anns = []
    for ann in anns:
        bbox = ann.get("bbox") or []
        if ann.get("image_id") not in image_ids or ann.get("category_id") not in category_ids:
            result["removed_annotations"] += 1
            continue
        if len(bbox) != 4 or float(bbox[2]) <= 0 or float(bbox[3]) <= 0:
            result["removed_annotations"] += 1
            continue
        ann.setdefault("area", float(bbox[2]) * float(bbox[3]))
        ann.setdefault("iscrowd", 0)
        valid_anns.append(ann)
    existing = {p.name for p in split_dir.iterdir() if p.is_file()}
    missing_images = [img.get("file_name") for img in images if img.get("file_name") not in existing]
    if missing_images:
        raise RuntimeError(f"{split_dir.name} annotations reference missing image files. First missing: {missing_images[0]}")
    data["annotations"] = valid_anns
    ann_path.write_text(json.dumps(data, indent=2), encoding="utf-8")
    result.update({"images": len(images), "annotations": len(valid_anns), "categories": len(categories)})
    return result


def _validate_detection_dataset(dataset_dir: Path):
    reports = []
    train_dir = dataset_dir / "train"
    valid_dir = dataset_dir / "valid"
    if not valid_dir.exists() and (dataset_dir / "val").exists():
        shutil.copytree(dataset_dir / "val", valid_dir, dirs_exist_ok=True)
    for split in (train_dir, valid_dir):
        reports.append(_validate_coco_split(split))
    if reports[0]["images"] <= 0:
        raise RuntimeError("Training split has no images.")
    if reports[1]["images"] <= 0:
        raise RuntimeError("Validation split has no images. RF-DETR requires a non-empty validation split.")
    return reports


def _validate_classification_dataset(dataset_dir: Path):
    train = dataset_dir / "train"
    if not train.exists():
        raise RuntimeError(f"Missing classification train directory: {train}")
    classes = [p for p in train.iterdir() if p.is_dir()]
    if not classes:
        raise RuntimeError("Classification train directory has no class folders.")
    return [{"split": "train", "images": _count_images(train), "categories": len(classes), "annotations": 0, "removed_annotations": 0}]


_METRIC_KEY_NORMALIZE = re.compile(r"[^a-z0-9]+")


def _normalize_metric_key(key: str) -> str:
    return _METRIC_KEY_NORMALIZE.sub("_", str(key).strip().lower()).strip("_")


def _first_numeric(metrics: dict, keys) -> float | None:
    if not isinstance(metrics, dict):
        return None
    normalized = {_normalize_metric_key(k): v for k, v in metrics.items()}
    for key in keys:
        value = metrics.get(key)
        if value is None:
            value = normalized.get(_normalize_metric_key(key))
        if isinstance(value, (int, float)):
            return float(value)
        if isinstance(value, str):
            text = value.strip().rstrip("%")
            try:
                number = float(text)
            except ValueError:
                continue
            if value.strip().endswith("%"):
                number /= 100.0
            return number
    return None


def _format_hms(seconds: float) -> str:
    seconds = max(0, int(seconds))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h}h {m:02d}m {s:02d}s"


class VisoLabelColabGUI:
    """Modern card-based UI for the VisoLabel Colab trainer."""

    PROGRESS_CARD_ID = "vl-progress-card"
    METRIC_MAP_ID = "vl-metric-map"
    METRIC_AP50_ID = "vl-metric-ap50"
    METRIC_THROUGHPUT_ID = "vl-metric-throughput"
    SUMMARY_CARDS_ID = "vl-summary-cards"

    def __init__(self):
        self.session = requests.Session()
        self.token_value = ""
        self.api_base = DEFAULT_API_BASE_URL
        self.api_token = ""
        self.manifest = None
        self.cfg = None
        self.task = ""
        self.command = ""
        self.dataset_dir = None
        self.training_finished = False
        self.started_at = None
        self.uploaded_artifacts = {}
        self.current_epoch = 0
        self.total_epochs = 0
        self.current_iteration = 0
        self.total_iterations = 0
        self.training_started_at = None
        self.latest_loss = None
        self.latest_map = None
        self.best_ap50 = None
        self.best_ap50_epoch = None
        self.latest_throughput = None
        self._last_epoch_upload_at = 0.0
        self._artifact_finalize_warning_logged = False
        self._uploader_thread = None
        self._uploader_stop = None
        self._upload_lock = threading.Lock()
        self._metrics_thread = None
        self._metrics_stop = None
        self._progress_thread = None
        self._progress_stop = None
        self._metrics_seen_mtime = 0.0
        self._tick_thread = None
        self._tick_stop = None
        self._last_live_log_line = ""

        self.token = widgets.Password(
            description="",
            placeholder="Paste your VisoLabel Colab token",
            layout=widgets.Layout(width="100%", height="42px"),
        )

        self.run_button = widgets.Button(
            description="Run full pipeline",
            icon="play",
            button_style="primary",
            layout=widgets.Layout(width="190px", height="42px"),
        )
        self.resolve_button = widgets.Button(
            description="Resolve bundle",
            layout=widgets.Layout(width="160px", height="42px"),
        )
        self.train_button = widgets.Button(
            description="Start training",
            layout=widgets.Layout(width="160px", height="42px"),
            disabled=True,
        )
        self.upload_button = widgets.Button(
            description="Upload results",
            layout=widgets.Layout(width="160px", height="42px"),
            disabled=True,
        )
        self.clear_button = widgets.Button(
            description="Clear logs",
            layout=widgets.Layout(width="120px", height="42px"),
        )

        self.status = widgets.HTML()
        self.progress_card = widgets.HTML()
        self.live_log = widgets.HTML()
        self.summary = widgets.HTML()
        self.log_output = widgets.Output()

        self.run_button.on_click(self._on_run_full)
        self.resolve_button.on_click(self._on_resolve)
        self.train_button.on_click(self._on_train)
        self.upload_button.on_click(self._on_upload)
        self.clear_button.on_click(self._on_clear_logs)

        self._set_status("Ready", "Paste your VisoLabel Colab token and click Run full pipeline.", "")
        self._render_progress_card()
        self._set_live_log("Waiting for training output.")
        self._render_summary()

    # --------------------------------------------------------------- display

    def display(self):
        display(HTML(self._stylesheet()))
        header = widgets.HTML(
            "<div class='vl-hero'>"
            "<div class='vl-hero-icon'>VL</div>"
            "<div class='vl-hero-text'>"
            "<div class='vl-hero-title'>VisoLabel Colab Trainer</div>"
            "<div class='vl-hero-subtitle'>Download &middot; decrypt &middot; validate &middot; train &middot; upload</div>"
            "</div>"
            "</div>"
        )
        token_section = widgets.HTML("<div class='vl-section-label'>Colab token</div>")
        button_row = widgets.HBox(
            [self.run_button, self.resolve_button, self.train_button, self.upload_button, self.clear_button],
            layout=widgets.Layout(gap="10px", flex_flow="row wrap", margin="4px 0 4px 0"),
        )
        live_log_section = widgets.HTML("<div class='vl-section-label' style='margin-top:6px;'>Realtime log</div>")
        log_section = widgets.HTML("<div class='vl-section-label' style='margin-top:6px;'>Full log</div>")
        log_box = widgets.Box(
            [self.log_output],
            layout=widgets.Layout(
                border="1px solid #2a3142",
                border_radius="12px",
                padding="10px 14px",
                background_color="#0f1422",
                max_height="360px",
                overflow_y="auto",
                width="100%",
            ),
        )
        display(widgets.VBox(
            [
                header,
                token_section,
                self.token,
                button_row,
                self.status,
                self.progress_card,
                live_log_section,
                self.live_log,
                self.summary,
                log_section,
                log_box,
            ],
            layout=widgets.Layout(gap="10px"),
        ))

    def _stylesheet(self) -> str:
        return """
        <style>
          .vl-hero {
            display:flex; align-items:center; gap:14px;
            padding:18px 22px; border-radius:14px; margin-bottom:6px;
            background:linear-gradient(135deg,#1f2a44 0%,#2d3f6b 60%,#3b4f8a 100%);
            color:#f4f7ff; box-shadow:0 4px 18px rgba(31,42,68,0.25);
          }
          .vl-hero-icon {
            width:44px; height:44px; border-radius:12px;
            background:rgba(255,255,255,0.14);
            display:flex; align-items:center; justify-content:center;
            font-weight:800; font-size:16px; letter-spacing:1px;
          }
          .vl-hero-title { font-size:20px; font-weight:700; }
          .vl-hero-subtitle { font-size:12px; opacity:0.85; margin-top:2px; }
          .vl-section-label {
            font-size:11px; text-transform:uppercase; letter-spacing:1.2px;
            color:#5b6580; font-weight:700; margin-top:4px;
          }
          .vl-status {
            border-radius:12px; padding:12px 16px; border:1px solid #2a3142;
            background:#141b2d; color:#d6dcef;
          }
          .vl-status h3 { font-size:14px; margin:0 0 3px 0; font-weight:700; }
          .vl-status p { margin:0; color:#8d97b3; font-size:12px; }
          .vl-status.ok { border-color:#1f6f4a; background:#0e2a1f; }
          .vl-status.ok h3 { color:#5be59b; }
          .vl-status.err { border-color:#7a2a36; background:#2a0f15; }
          .vl-status.err h3 { color:#ff6b85; }
          .vl-status.active { border-color:#2b4d8a; background:#0f1d3a; }
          .vl-status.active h3 { color:#7ab2ff; }

          .vl-card {
            background:#131a2c; border:1px solid #2a3142; border-radius:14px;
            padding:18px 20px; color:#e9edf7;
            box-shadow:0 1px 0 rgba(255,255,255,0.02) inset;
          }
          .vl-progress-head { display:flex; justify-content:space-between; align-items:center; }
          .vl-progress-title { font-size:13px; font-weight:700; color:#e9edf7; }
          .vl-progress-epoch { font-size:12px; font-weight:600; color:#8d97b3; }
          .vl-percent-row { display:flex; align-items:baseline; gap:10px; margin-top:10px; }
          .vl-percent-big { font-size:38px; font-weight:800; color:#ffffff; line-height:1; }
          .vl-percent-chip {
            font-size:11px; font-weight:700; letter-spacing:0.4px;
            background:#1f2a44; color:#8db4ff;
            border:1px solid #2c3d6a; border-radius:999px;
            padding:3px 10px;
          }
          .vl-progress-track {
            margin-top:12px; height:8px; border-radius:6px;
            background:#1a223a; overflow:hidden;
          }
          .vl-progress-fill {
            height:100%; border-radius:6px;
            background:linear-gradient(90deg,#7ab2ff 0%,#5d8cff 100%);
            transition:width 320ms ease-out;
          }
          .vl-iter-row {
            margin-top:14px; display:flex; justify-content:space-between; gap:10px;
            font-size:12px; color:#8d97b3; font-weight:600;
          }
          .vl-iter-track {
            margin-top:6px; height:5px; border-radius:4px;
            background:#1a223a; overflow:hidden;
          }
          .vl-iter-fill {
            height:100%; border-radius:4px;
            background:linear-gradient(90deg,#5be59b 0%,#3fbf7e 100%);
            transition:width 320ms ease-out;
          }
          .vl-stat-row {
            display:grid; grid-template-columns:repeat(3,1fr);
            gap:14px; margin-top:16px;
          }
          .vl-stat {
            background:#0f1422; border:1px solid #232a3d; border-radius:10px;
            padding:9px 12px;
          }
          .vl-stat-key {
            font-size:10px; text-transform:uppercase; letter-spacing:1.1px;
            color:#6b7793; font-weight:700;
          }
          .vl-stat-value {
            font-size:14px; font-weight:700; color:#f0f3ff; margin-top:3px;
          }

          .vl-metrics {
            display:grid; gap:12px;
            grid-template-columns:repeat(auto-fit,minmax(160px,1fr));
            margin-top:6px;
          }
          .vl-metric {
            background:#131a2c; border:1px solid #2a3142; border-radius:12px;
            padding:14px 16px;
          }
          .vl-metric-label {
            font-size:12px; font-weight:700; color:#8d97b3;
          }
          .vl-metric-value {
            font-size:24px; font-weight:800; color:#ffffff; margin-top:6px;
          }
          .vl-metric-sub {
            font-size:11px; color:#6b7793; margin-top:4px;
          }

          .vl-summary {
            display:grid; gap:10px;
            grid-template-columns:repeat(auto-fit,minmax(160px,1fr));
            margin-top:6px;
          }
          .vl-sum-card {
            background:#131a2c; border:1px solid #2a3142; border-radius:10px;
            padding:10px 12px;
          }
          .vl-sum-label {
            font-size:11px; text-transform:uppercase; letter-spacing:1px;
            color:#6b7793; font-weight:700;
          }
          .vl-sum-value {
            font-size:13px; font-weight:700; color:#f0f3ff; margin-top:3px;
            overflow-wrap:anywhere;
          }
          .vl-sum-sub {
            font-size:11px; color:#7a82a0; margin-top:2px; overflow-wrap:anywhere;
          }
          .vl-live-log {
            display:flex; align-items:center; gap:10px; min-height:38px;
            background:#0f1422; border:1px solid #2a3142; border-radius:10px;
            padding:10px 12px; color:#d6dcef;
          }
          .vl-live-log-label {
            flex:0 0 auto; font-size:10px; text-transform:uppercase;
            letter-spacing:1px; color:#6b7793; font-weight:800;
          }
          .vl-live-log-line {
            flex:1 1 auto; min-width:0; font-family:ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
            font-size:12px; color:#f0f3ff; white-space:nowrap; overflow:hidden; text-overflow:ellipsis;
          }
        </style>
        """

    def _set_status(self, title: str, detail: str, kind: str):
        cls = ("vl-status " + (_html.escape(kind) if kind else "")).strip()
        self.status.value = (
            f"<div class='{cls}'>"
            f"<h3>{_html.escape(title)}</h3>"
            f"<p>{_html.escape(detail)}</p>"
            f"</div>"
        )

    def _set_live_log(self, line: str):
        text = str(line or "").strip() or "Waiting for training output."
        if len(text) > 600:
            text = "..." + text[-597:]
        self._last_live_log_line = text
        self.live_log.value = (
            "<div class='vl-live-log'>"
            "<div class='vl-live-log-label'>Last log line</div>"
            f"<div class='vl-live-log-line'>{_html.escape(text)}</div>"
            "</div>"
        )

    def _set_live_log_from_text(self, text: str):
        normalized = str(text or "").replace("\r", "\n")
        lines = [line.strip() for line in normalized.splitlines() if line.strip()]
        if lines:
            self._set_live_log(lines[-1])

    def _on_clear_logs(self, _=None):
        self.log_output.clear_output()
        self._set_live_log("Full log cleared. Waiting for next line.")

    def _log(self, message: str):
        ts = time.strftime("%H:%M:%S")
        self._set_live_log(message)
        with self.log_output:
            print(f"[{ts}] {message}", flush=True)

    def _read_training_log_tail(self, max_lines: int = 160) -> str:
        if not TRAINING_LOG.exists():
            return ""
        try:
            lines = TRAINING_LOG.read_text(encoding="utf-8", errors="replace").splitlines()
        except Exception:
            return ""
        return "\n".join(lines[-max_lines:])

    def _is_cuda_oom_text(self, text: str) -> bool:
        lower = str(text or "").lower()
        cuda_markers = (
            "cuda out of memory",
            "torch.cuda.outofmemoryerror",
            "cuda error: out of memory",
            "cublas_status_alloc_failed",
            "cudnn_status_alloc_failed",
        )
        if any(marker in lower for marker in cuda_markers):
            return True
        return "outofmemoryerror" in lower and ("cuda" in lower or "gpu" in lower)

    def _summarize_failure_from_log(self, text: str) -> str:
        cleaned = [re.sub(r"\s+", " ", line).strip() for line in str(text or "").splitlines()]
        cleaned = [line for line in cleaned if line]
        if not cleaned:
            return "No error details were written to the log."
        interesting = []
        markers = ("error", "exception", "traceback", "runtimeerror", "valueerror", "importerror", "modulenotfounderror", "failed")
        for line in reversed(cleaned):
            if any(marker in line.lower() for marker in markers):
                interesting.append(line)
            if len(interesting) >= 3:
                break
        if interesting:
            interesting.reverse()
            summary = " | ".join(interesting)
        else:
            summary = " | ".join(cleaned[-3:])
        if len(summary) > 700:
            summary = "..." + summary[-697:]
        return summary

    def _training_failure_message(self, code: int) -> str:
        tail = self._read_training_log_tail()
        if self._is_cuda_oom_text(tail):
            return (
                "CUDA out of memory. Colab's current GPU does not have enough VRAM for this training run. "
                "Upgrade to Colab Pro or Pro+ to get access to better GPUs, or decrease the batch size "
                "in VisoLabel and start a new Colab training run. "
                f"Full log: {TRAINING_LOG}."
            )
        summary = self._summarize_failure_from_log(tail)
        return f"Training failed with exit code {code}. {summary} Full log: {TRAINING_LOG}."

    # ---------------------------------------------------- progress rendering

    def _render_progress_card(self):
        total_epochs = self.total_epochs or int((self.cfg or {}).get("epochs", 0) or 0)
        epoch = self.current_epoch
        epoch_text = (
            f"Epoch {epoch} / {total_epochs}" if total_epochs and epoch
            else (f"Epoch — / {total_epochs}" if total_epochs else "Epoch — / —")
        )
        percent = 0.0
        if total_epochs > 0:
            percent = max(0.0, min(100.0, (epoch / total_epochs) * 100.0)) if epoch else 0.0

        iter_total = self.total_iterations
        iter_current = self.current_iteration
        iter_percent = (iter_current / iter_total) * 100.0 if iter_total else 0.0
        iter_text = (
            f"Iteration {iter_current} / {iter_total}" if iter_total
            else (f"Iteration {iter_current}" if iter_current else "Iteration — / —")
        )

        elapsed_seconds = (time.time() - self.training_started_at) if self.training_started_at else 0
        if elapsed_seconds > 0 and total_epochs and epoch:
            seconds_per_epoch = elapsed_seconds / max(epoch, 1)
            eta_seconds = max(total_epochs - epoch, 0) * seconds_per_epoch
            eta_text = _format_hms(eta_seconds)
        else:
            eta_text = "—"
        elapsed_text = _format_hms(elapsed_seconds) if elapsed_seconds else "0h 00m 00s"
        loss_text = f"{self.latest_loss:.3f}" if isinstance(self.latest_loss, (int, float)) else "—"

        self.progress_card.value = (
            "<div class='vl-card'>"
            "<div class='vl-progress-head'>"
            "<div class='vl-progress-title'>Training progress</div>"
            f"<div class='vl-progress-epoch'>{_html.escape(epoch_text)}</div>"
            "</div>"
            "<div class='vl-percent-row'>"
            f"<div class='vl-percent-big'>{percent:.1f}%</div>"
            "<div class='vl-percent-chip'>complete</div>"
            "</div>"
            f"<div class='vl-progress-track'><div class='vl-progress-fill' style='width:{percent:.2f}%;'></div></div>"
            "<div class='vl-iter-row'>"
            f"<span>{_html.escape(iter_text)}</span>"
            f"<span>{iter_percent:.0f}%</span>"
            "</div>"
            f"<div class='vl-iter-track'><div class='vl-iter-fill' style='width:{iter_percent:.2f}%;'></div></div>"
            "<div class='vl-stat-row'>"
            f"<div class='vl-stat'><div class='vl-stat-key'>Elapsed</div><div class='vl-stat-value'>{_html.escape(elapsed_text)}</div></div>"
            f"<div class='vl-stat'><div class='vl-stat-key'>ETA</div><div class='vl-stat-value'>{_html.escape(eta_text)}</div></div>"
            f"<div class='vl-stat'><div class='vl-stat-key'>Loss</div><div class='vl-stat-value'>{_html.escape(loss_text)}</div></div>"
            "</div>"
            "</div>"
            + self._metric_cards_html()
        )

    def _metric_cards_html(self) -> str:
        if self.task == "classification":
            primary_label = "Accuracy"
            primary_value = (
                f"{self.latest_map:.1f}%" if isinstance(self.latest_map, (int, float))
                else "—"
            )
            secondary_label = "Best accuracy"
            secondary_value = (
                f"{self.best_ap50:.1f}%" if isinstance(self.best_ap50, (int, float))
                else "—"
            )
            secondary_sub = (
                f"epoch {self.best_ap50_epoch}" if self.best_ap50_epoch else ""
            )
        else:
            primary_label = "mAP"
            primary_value = (
                f"{self.latest_map:.3f}" if isinstance(self.latest_map, (int, float))
                else "—"
            )
            secondary_label = "Best AP50"
            secondary_value = (
                f"{self.best_ap50:.3f}" if isinstance(self.best_ap50, (int, float))
                else "—"
            )
            secondary_sub = (
                f"epoch {self.best_ap50_epoch}" if self.best_ap50_epoch else ""
            )

        throughput_value = (
            f"{self.latest_throughput:.1f}" if isinstance(self.latest_throughput, (int, float))
            else "—"
        )
        throughput_sub = "img/s" if isinstance(self.latest_throughput, (int, float)) else ""

        return (
            "<div class='vl-section-label' style='margin-top:14px;'>Live metrics</div>"
            "<div class='vl-metrics'>"
            f"<div class='vl-metric'><div class='vl-metric-label'>{primary_label}</div>"
            f"<div class='vl-metric-value'>{_html.escape(primary_value)}</div>"
            "<div class='vl-metric-sub'></div></div>"
            f"<div class='vl-metric'><div class='vl-metric-label'>{secondary_label}</div>"
            f"<div class='vl-metric-value'>{_html.escape(secondary_value)}</div>"
            f"<div class='vl-metric-sub'>{_html.escape(secondary_sub)}</div></div>"
            "<div class='vl-metric'><div class='vl-metric-label'>Throughput</div>"
            f"<div class='vl-metric-value'>{_html.escape(throughput_value)}</div>"
            f"<div class='vl-metric-sub'>{_html.escape(throughput_sub)}</div></div>"
            "</div>"
        )

    def _render_summary(self):
        cfg = self.cfg or {}
        classes = cfg.get("classes") or []
        if not isinstance(classes, list):
            classes = []
        values = [
            ("Task", self.task or "—", self.command or "Waiting for token"),
            ("Model", str(cfg.get("model_name", "—")), f"{len(classes)} classes" if classes else ""),
            ("Epochs", str(cfg.get("epochs", "—")), f"Batch {cfg.get('batch_size', '—')}"),
            ("Image size", str(cfg.get("image_size", cfg.get("resolution", "—"))), "Adjusted to model multiples if needed"),
            ("Dataset", f"{_count_images(self.dataset_dir) if self.dataset_dir else 0} images", str(self.dataset_dir or "—")),
        ]
        cards = ["<div class='vl-section-label' style='margin-top:10px;'>Run summary</div><div class='vl-summary'>"]
        for label, value, sub in values:
            cards.append(
                "<div class='vl-sum-card'>"
                f"<div class='vl-sum-label'>{_html.escape(label)}</div>"
                f"<div class='vl-sum-value'>{_html.escape(str(value))}</div>"
                f"<div class='vl-sum-sub'>{_html.escape(str(sub))}</div>"
                "</div>"
            )
        cards.append("</div>")
        self.summary.value = "".join(cards)

    # ------------------------------------------------------- network helpers

    def _post(self, endpoint: str, payload: dict) -> dict:
        url = _normalize_url(self.api_base, endpoint)
        response = self.session.post(url, json=payload, timeout=120)
        response.raise_for_status()
        return response.json()

    def _download(self, url: str, destination: Path):
        destination.parent.mkdir(parents=True, exist_ok=True)
        with self.session.get(url, stream=True, timeout=120) as response:
            response.raise_for_status()
            total = int(response.headers.get("content-length") or 0)
            with open(destination, "wb") as f:
                iterator = response.iter_content(chunk_size=1024 * 1024)
                if total:
                    iterator = tqdm(iterator, total=max(math.ceil(total / (1024 * 1024)), 1), unit="MB", desc=destination.name)
                for chunk in iterator:
                    if not chunk:
                        continue
                    f.write(chunk)
        self._log(f"Downloaded {destination.name}: {_format_bytes(destination.stat().st_size)}")

    def _decrypt_bundle(self, encrypted_path: Path, output_path: Path, password: str, encryption: dict):
        if encryption.get("algorithm") != "AES-256-GCM":
            raise RuntimeError(f"Unsupported encryption algorithm: {encryption.get('algorithm')}")
        if encryption.get("kdf") != "PBKDF2-HMAC-SHA256":
            raise RuntimeError(f"Unsupported encryption KDF: {encryption.get('kdf')}")
        nonce = base64.b64decode(str(encryption["nonce_b64"]).encode("ascii"), validate=True)
        tag = base64.b64decode(str(encryption["tag_b64"]).encode("ascii"), validate=True)
        salt = base64.b64decode(str(encryption["salt_b64"]).encode("ascii"), validate=True)
        aad_value = encryption.get("aad_b64", "")
        aad = base64.b64decode(str(aad_value).encode("ascii"), validate=True) if aad_value else b"visolabel-colab-bundle-v1"
        iterations = int(encryption.get("kdf_iterations") or encryption.get("iterations") or 200000)
        kdf = PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=salt, iterations=iterations)
        key = kdf.derive(password.encode("utf-8"))
        decryptor = Cipher(algorithms.AES(key), modes.GCM(nonce, tag)).decryptor()
        decryptor.authenticate_additional_data(aad)
        with open(encrypted_path, "rb") as src, open(output_path, "wb") as dst:
            for chunk in iter(lambda: src.read(1024 * 1024), b""):
                data = decryptor.update(chunk)
                if data:
                    dst.write(data)
            data = decryptor.finalize()
            if data:
                dst.write(data)

    def _extract_manifest(self):
        if BUNDLE_DIR.exists():
            shutil.rmtree(BUNDLE_DIR)
        BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
        _safe_extract_zip(BUNDLE_ZIP, BUNDLE_DIR)
        manifest_path = BUNDLE_DIR / "training_bundle.json"
        if not manifest_path.exists():
            raise RuntimeError("Bundle is missing training_bundle.json.")
        self.manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        self.task = self.manifest["task"]["task_type"]
        self.command = self.manifest["task"]["command"]
        self.cfg = self.manifest["training_config"]
        self.total_epochs = int(self.cfg.get("epochs", 0) or 0)
        self.dataset_dir = BUNDLE_DIR / self.manifest["paths"]["dataset_dir"]
        if self.command == "train":
            reports = _validate_detection_dataset(self.dataset_dir)
        else:
            reports = _validate_classification_dataset(self.dataset_dir)
        (OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
        (OUTPUT_DIR / "dataset_validation.json").write_text(json.dumps(reports, indent=2), encoding="utf-8")
        for report in reports:
            self._log(f"Validated {report['split']}: {report['images']} images, {report['annotations']} annotations, {report['categories']} categories, removed {report['removed_annotations']}.")
        self.train_button.disabled = False
        self._render_summary()
        self._render_progress_card()

    def resolve_bundle(self):
        self.token_value = self.token.value.strip()
        if not self.token_value:
            raise RuntimeError("Paste a VisoLabel Colab token first.")
        self.api_base = DEFAULT_API_BASE_URL
        self.started_at = time.time()
        self.training_finished = False
        self.uploaded_artifacts = {}
        self.current_epoch = 0
        self.current_iteration = 0
        self.total_iterations = 0
        self.training_started_at = None
        self.latest_loss = None
        self.latest_map = None
        self.best_ap50 = None
        self.best_ap50_epoch = None
        self.latest_throughput = None
        self._last_epoch_upload_at = 0.0
        self._stop_artifact_uploader()
        self.upload_button.disabled = True
        WORK_DIR.mkdir(parents=True, exist_ok=True)
        for path in (BUNDLE_ENCRYPTED, BUNDLE_ZIP):
            if path.exists():
                path.unlink()
        self._set_status("Resolving bundle", "Downloading bundle metadata and preparing the dataset.", "active")
        self._log("Resolving token.")
        if self.token_value.startswith(TOKEN_PREFIX) or self.token_value.startswith(LEGACY_TOKEN_PREFIX):
            parsed = _parse_encrypted_token(self.token_value)
            self.api_token = parsed["server_token"]
            if parsed["api_url"]:
                self.api_base = parsed["api_url"]
            bundle_url = parsed["bundle_url"]
            encryption = parsed["encryption"]
            password = parsed["password"]
            if not bundle_url or not encryption:
                payload = {"token": self.api_token}
                if parsed["bundle_id"]:
                    payload["bundle_id"] = parsed["bundle_id"]
                resolved = self._post("/api/v1/colab/resolve-token/", payload)
                bundle_url = bundle_url or _extract_bundle_url(resolved)
                encryption = encryption or _extract_encryption_metadata(resolved)
            if not bundle_url or not password or not encryption:
                raise RuntimeError("Encrypted token could not resolve bundle URL, password, or encryption metadata. Regenerate the token with the latest VisoLabel build.")
            self._download(bundle_url, BUNDLE_ENCRYPTED)
            expected_cipher_sha = encryption.get("ciphertext_sha256", "")
            if expected_cipher_sha and _sha256(BUNDLE_ENCRYPTED) != expected_cipher_sha:
                raise RuntimeError("Encrypted bundle SHA-256 does not match token metadata.")
            self._set_status("Decrypting bundle", "The dataset is decrypted locally in this Colab runtime.", "active")
            self._log("Decrypting bundle locally.")
            self._decrypt_bundle(BUNDLE_ENCRYPTED, BUNDLE_ZIP, password, encryption)
            expected_plain_sha = encryption.get("plaintext_sha256", "")
            if expected_plain_sha and _sha256(BUNDLE_ZIP) != expected_plain_sha:
                raise RuntimeError("Decrypted bundle SHA-256 does not match token metadata.")
        else:
            self.api_token = self.token_value
            resolved = self._post("/api/v1/colab/resolve-token/", {"token": self.token_value})
            bundle_url = _extract_bundle_url(resolved)
            if not bundle_url:
                raise RuntimeError("bundle_url missing in resolve-token response.")
            self._download(bundle_url, BUNDLE_ZIP)
        self._log("Extracting and validating bundle.")
        self._extract_manifest()
        self._set_status("Bundle ready", "Review the run summary below, then start training.", "ok")

    def _install_training_deps(self):
        if self.command == "train":
            packages = ["rfdetr[train,loggers]", "pycocotools", "supervision"]
        else:
            packages = ["timm", "torchvision"]
        self._set_status("Installing dependencies", "Colab may take a few minutes the first time.", "active")
        self._log("Installing: " + " ".join(packages))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])

    def train(self):
        if not self.manifest:
            self.resolve_bundle()
        if OUTPUT_DIR.exists():
            shutil.rmtree(OUTPUT_DIR)
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        self._install_training_deps()
        RUNNER_PATH.write_text(RUNNER_CODE, encoding="utf-8")
        self.training_started_at = time.time()
        self._set_status("Training", "Live progress is shown in the card below. Keep this tab open.", "active")
        self._set_live_log("Starting training subprocess...")
        self._render_progress_card()
        manifest_path = BUNDLE_DIR / "training_bundle.json"
        command = [sys.executable, "-u", str(RUNNER_PATH), str(manifest_path), str(self.dataset_dir), str(OUTPUT_DIR)]
        env = dict(os.environ)
        env["PYTHONUNBUFFERED"] = "1"
        self._start_artifact_uploader()
        self._start_metrics_watcher()
        self._start_progress_watcher()
        self._start_tick_thread()
        try:
            with open(TRAINING_LOG, "w", encoding="utf-8") as log_file:
                def stream_training_text(text):
                    if not text:
                        return
                    log_file.write(text.replace("\r", "\n"))
                    log_file.flush()
                    self._set_live_log_from_text(text)
                    with self.log_output:
                        print(text, end="", flush=True)
                    self._absorb_text(text)
                    self._maybe_upload_epoch_artifacts()
                    self._absorb_progress_file()

                if os.name == "posix":
                    import pty
                    import select
                    master_fd, slave_fd = pty.openpty()
                    proc = subprocess.Popen(command, stdout=slave_fd, stderr=slave_fd, stdin=subprocess.DEVNULL, env=env, close_fds=True)
                    os.close(slave_fd)
                    try:
                        while True:
                            ready, _, _ = select.select([master_fd], [], [], 0.2)
                            if ready:
                                try:
                                    data = os.read(master_fd, 4096)
                                except OSError:
                                    break
                                if not data:
                                    break
                                stream_training_text(data.decode("utf-8", errors="replace"))
                            elif proc.poll() is not None:
                                break
                    finally:
                        os.close(master_fd)
                    code = proc.wait()
                else:
                    proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
                    for raw in iter(proc.stdout.readline, ""):
                        stream_training_text(raw)
                    code = proc.wait()
        finally:
            self._stop_artifact_uploader()
            self._stop_metrics_watcher()
            self._stop_progress_watcher()
            self._stop_tick_thread()
        if code != 0:
            message = self._training_failure_message(code)
            try:
                with open(TRAINING_LOG, "a", encoding="utf-8") as log_file:
                    log_file.write(f"\n[visolabel-colab] {message}\n")
            except Exception:
                pass
            self._upload_pending_artifacts(run_status="failed", raise_errors=False)
            self._set_live_log(message)
            raise RuntimeError(message)
        self._absorb_metrics_file()
        self.training_finished = True
        if self.total_epochs:
            self.current_epoch = max(self.current_epoch, self.total_epochs)
        self._render_progress_card()
        self._upload_pending_artifacts(run_status="completed", raise_errors=False)
        self.upload_button.disabled = False
        self._set_status("Training finished", f"Outputs are in {OUTPUT_DIR}.", "ok")
        self._log(f"Training finished. Output dir: {OUTPUT_DIR}")

    # ----------------------------------------------------- progress sources

    def _absorb_text(self, text: str):
        """Fallback parser for stdout when progress.json is not (yet) available."""
        changed = False
        # PyTorch Lightning prints "Epoch N:" where N is 0-indexed. Treat that
        # as the (1-indexed) current epoch.
        for match in re.finditer(r"Epoch\s+(\d+)\s*:", text):
            epoch = int(match.group(1)) + 1
            if epoch > self.current_epoch:
                self.current_epoch = epoch
                changed = True
        # Explicit "Epoch X/Y" format used by some trainers.
        for match in re.finditer(r"Epoch\s+(\d+)\s*/\s*(\d+)", text):
            epoch = int(match.group(1))
            total = int(match.group(2))
            if epoch > self.current_epoch:
                self.current_epoch = epoch
                changed = True
            if total and total != self.total_epochs:
                self.total_epochs = total
                changed = True
        # tqdm-style iteration: "  10%|███| 25/250 [00:05<00:48, loss=...]".
        for match in re.finditer(r"(\d+)\s*/\s*(\d+)\s*\[[^\]]*\]", text):
            current = int(match.group(1))
            total = int(match.group(2))
            if total > 1 and total != self.total_epochs:
                if current <= total:
                    self.current_iteration = current
                    self.total_iterations = total
                    changed = True
        loss_match = re.search(r"loss[=: ]\s*([0-9]+\.[0-9]+)", text, re.IGNORECASE)
        if loss_match:
            try:
                self.latest_loss = float(loss_match.group(1))
                changed = True
            except ValueError:
                pass
        if changed:
            self._render_progress_card()

    def _absorb_metrics_file(self):
        if not METRICS_PATH.exists():
            return
        try:
            stat = METRICS_PATH.stat()
        except OSError:
            return
        mtime = float(stat.st_mtime)
        if mtime <= self._metrics_seen_mtime:
            return
        self._metrics_seen_mtime = mtime
        try:
            data = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
        except Exception:
            return
        latest = data.get("latest") if isinstance(data, dict) else None
        if not isinstance(latest, dict):
            return
        history = data.get("history") if isinstance(data, dict) else []

        for row in history if isinstance(history, list) else []:
            if isinstance(row, dict):
                self._update_best_ap50(row)
        self._update_best_ap50(latest)

        loss = _first_numeric(latest, (
            "loss", "train_loss", "total_loss", "train/loss", "loss_total",
            "train_total_loss", "loss_ce", "box_loss", "cls_loss", "dfl_loss",
        ))
        if loss is not None:
            self.latest_loss = loss

        if self.task == "classification":
            acc = _first_numeric(latest, ("accuracy", "val_accuracy", "acc", "top1"))
            if acc is not None:
                self.latest_map = acc * 100.0 if 0.0 <= acc <= 1.0 else acc
        else:
            map_value = _first_numeric(latest, (
                "mAP", "map", "map_50_95", "map_5095", "mAP@50:95",
                "metrics/mAP50-95(B)", "metrics/mAP50-95(M)", "val_map", "val/mAP",
                "coco_eval_bbox", "coco_eval_masks",
            ))
            if map_value is not None:
                self.latest_map = map_value

        throughput = _first_numeric(latest, (
            "throughput", "ips", "imgs_per_sec", "images_per_second", "fps",
            "img_s", "imgs_s", "images_s", "samples_per_second", "it_s",
        ))
        if throughput is not None:
            self.latest_throughput = throughput

        epoch = _first_numeric(latest, ("epoch", "current_epoch", "epochs_completed"))
        if epoch is not None:
            self.current_epoch = max(self.current_epoch, int(epoch))
        total = _first_numeric(latest, ("total_epochs", "epochs", "max_epoch", "num_epochs"))
        if total is not None and int(total) > 0:
            self.total_epochs = max(self.total_epochs, int(total))

        self._render_progress_card()

    def _update_best_ap50(self, row: dict):
        if not isinstance(row, dict):
            return
        if self.task == "classification":
            ap = _first_numeric(row, ("accuracy", "val_accuracy", "acc", "top1"))
            if ap is None:
                return
            ap = ap * 100.0 if 0.0 <= ap <= 1.0 else ap
        else:
            ap = _first_numeric(row, ("AP50", "ap50", "map_50", "mAP50", "mAP@50",
                                     "metrics/mAP50(B)", "metrics/mAP50(M)"))
            if ap is None:
                return
        epoch = _first_numeric(row, ("epoch", "current_epoch", "epochs_completed"))
        if self.best_ap50 is None or ap > self.best_ap50:
            self.best_ap50 = ap
            self.best_ap50_epoch = int(epoch) if epoch is not None else self.current_epoch

    # --------------------------------------------------- background workers

    def _absorb_progress_file(self):
        progress_path = OUTPUT_DIR / "progress.json"
        if not progress_path.exists():
            return False
        try:
            data = json.loads(progress_path.read_text(encoding="utf-8"))
        except Exception:
            return False
        if not isinstance(data, dict):
            return False
        changed = False
        for key in ("current_epoch", "total_epochs", "current_iteration", "total_iterations"):
            value = data.get(key)
            if isinstance(value, (int, float)) and int(value) >= 0:
                if int(value) != getattr(self, key):
                    setattr(self, key, int(value))
                    changed = True
        loss = data.get("loss")
        if isinstance(loss, (int, float)):
            if self.latest_loss != float(loss):
                self.latest_loss = float(loss)
                changed = True
        throughput = data.get("throughput")
        if isinstance(throughput, (int, float)):
            if self.latest_throughput != float(throughput):
                self.latest_throughput = float(throughput)
                changed = True
        return changed

    def _start_progress_watcher(self):
        self._stop_progress_watcher()
        self._progress_stop = threading.Event()

        def watcher():
            while self._progress_stop and not self._progress_stop.wait(0.7):
                try:
                    if self._absorb_progress_file():
                        self._render_progress_card()
                except Exception:
                    pass

        self._progress_thread = threading.Thread(target=watcher, name="vl-progress-watcher", daemon=True)
        self._progress_thread.start()

    def _stop_progress_watcher(self):
        stop_event = getattr(self, "_progress_stop", None)
        thread = getattr(self, "_progress_thread", None)
        if stop_event is not None:
            stop_event.set()
        if thread is not None and thread.is_alive():
            thread.join(timeout=10)
        self._progress_thread = None
        self._progress_stop = None

    def _start_metrics_watcher(self):
        self._stop_metrics_watcher()
        self._metrics_stop = threading.Event()

        def watcher():
            while self._metrics_stop and not self._metrics_stop.wait(3):
                try:
                    self._absorb_metrics_file()
                except Exception:
                    pass

        self._metrics_thread = threading.Thread(target=watcher, name="vl-metrics-watcher", daemon=True)
        self._metrics_thread.start()

    def _stop_metrics_watcher(self):
        if self._metrics_stop is not None:
            self._metrics_stop.set()
        if self._metrics_thread is not None and self._metrics_thread.is_alive():
            self._metrics_thread.join(timeout=10)
        self._metrics_thread = None
        self._metrics_stop = None

    def _start_tick_thread(self):
        self._stop_tick_thread()
        self._tick_stop = threading.Event()

        def ticker():
            while self._tick_stop and not self._tick_stop.wait(1):
                try:
                    self._render_progress_card()
                except Exception:
                    pass

        self._tick_thread = threading.Thread(target=ticker, name="vl-tick", daemon=True)
        self._tick_thread.start()

    def _stop_tick_thread(self):
        if self._tick_stop is not None:
            self._tick_stop.set()
        if self._tick_thread is not None and self._tick_thread.is_alive():
            self._tick_thread.join(timeout=5)
        self._tick_thread = None
        self._tick_stop = None

    # -------------------------------------------------- artifact uploading

    def _artifact_relative_path(self, path: Path) -> str:
        try:
            return path.relative_to(OUTPUT_DIR).as_posix()
        except ValueError:
            return path.name

    def _artifact_signature(self, path: Path):
        stat = path.stat()
        return (int(stat.st_size), int(getattr(stat, "st_mtime_ns", int(stat.st_mtime * 1_000_000_000))))

    def _collect_uploadable_artifacts(self):
        artifacts = []
        if TRAINING_LOG.exists():
            artifacts.append(TRAINING_LOG)
        if OUTPUT_DIR.exists():
            artifacts.extend(sorted(
                p for p in OUTPUT_DIR.rglob("*")
                if p.is_file() and p.suffix.lower() in ARTIFACT_EXTENSIONS and p != TRAINING_LOG
            ))
        seen = set()
        unique = []
        for path in artifacts:
            key = str(path)
            if key in seen:
                continue
            seen.add(key)
            unique.append(path)
        return unique

    def _request_upload(self, path: Path, file_sha256: str, run_status: str) -> dict:
        return self._post("/api/v1/colab/request-checkpoint-upload/", {
            "token": self.api_token,
            "file_name": path.name,
            "relative_path": self._artifact_relative_path(path),
            "file_size": path.stat().st_size,
            "file_sha256": file_sha256,
            "content_type": "application/octet-stream",
            "task_type": self.task,
            "epoch": self.current_epoch or None,
            "run_status": run_status,
        })

    def _finalize_artifact_upload(self, path: Path, info: dict, file_sha256: str, response, run_status: str):
        payload = {
            "token": self.api_token,
            "upload_id": str(info.get("upload_id") or ""),
            "object_key": str(info.get("object_key") or info.get("key") or ""),
            "file_name": path.name,
            "relative_path": self._artifact_relative_path(path),
            "file_size": path.stat().st_size,
            "file_sha256": file_sha256,
            "content_type": "application/octet-stream",
            "task_type": self.task,
            "epoch": self.current_epoch or None,
            "run_status": run_status,
            "etag": response.headers.get("ETag", "") if hasattr(response, "headers") else "",
        }
        try:
            self._post("/api/v1/colab/finalize-checkpoint-upload/", payload)
        except Exception as exc:
            if not self._artifact_finalize_warning_logged:
                self._log(f"Artifact finalize endpoint not available: {exc}")
                self._artifact_finalize_warning_logged = True

    def _upload_artifact(self, path: Path, run_status: str = "training"):
        file_sha256 = _sha256(path)
        info = self._request_upload(path, file_sha256, run_status)
        upload_url = str(info.get("upload_url") or info.get("url") or "")
        if not upload_url:
            raise RuntimeError(f"Upload URL missing for {path.name}.")
        method = str(info.get("upload_method") or info.get("method") or "PUT").upper()
        headers = dict(info.get("upload_headers") or info.get("headers") or {})
        headers.pop("Host", None)
        self._log(f"Uploading {self._artifact_relative_path(path)} ({_format_bytes(path.stat().st_size)}).")
        with open(path, "rb") as f:
            if method == "POST":
                response = requests.post(upload_url, data=f, headers=headers, timeout=600)
            else:
                response = requests.put(upload_url, data=f, headers=headers, timeout=600)
        response.raise_for_status()
        self._finalize_artifact_upload(path, info, file_sha256, response, run_status)
        self.uploaded_artifacts[self._artifact_relative_path(path)] = self._artifact_signature(path)

    def _upload_pending_artifacts(self, run_status: str = "training", raise_errors: bool = False) -> int:
        if not self.api_token:
            return 0
        acquired = self._upload_lock.acquire(blocking=bool(raise_errors))
        if not acquired:
            return 0
        try:
            candidates = self._collect_uploadable_artifacts()
            pending = []
            for path in candidates:
                try:
                    signature = self._artifact_signature(path)
                except OSError:
                    continue
                if self.uploaded_artifacts.get(self._artifact_relative_path(path)) != signature:
                    pending.append(path)
            for path in pending:
                try:
                    self._upload_artifact(path, run_status=run_status)
                except Exception as exc:
                    if raise_errors:
                        raise
                    self._log(f"Artifact upload skipped for {path.name}: {exc}")
            return len(pending)
        finally:
            self._upload_lock.release()

    def _start_artifact_uploader(self):
        if not self.api_token:
            return
        self._stop_artifact_uploader()
        self._uploader_stop = threading.Event()

        def uploader_loop():
            self._log("Realtime artifact uploader started.")
            while self._uploader_stop and not self._uploader_stop.wait(5):
                if self.training_finished:
                    break
                uploaded = self._upload_pending_artifacts(run_status="training", raise_errors=False)
                if uploaded:
                    self._log(f"Realtime uploader sent {uploaded} updated artifact(s).")
            uploaded = self._upload_pending_artifacts(run_status="training", raise_errors=False)
            if uploaded:
                self._log(f"Realtime uploader sent {uploaded} final updated artifact(s).")
            self._log("Realtime artifact uploader stopped.")

        self._uploader_thread = threading.Thread(target=uploader_loop, name="vl-artifact-uploader", daemon=True)
        self._uploader_thread.start()

    def _stop_artifact_uploader(self):
        stop_event = self._uploader_stop
        thread = self._uploader_thread
        if stop_event is not None:
            stop_event.set()
        if thread is not None and thread.is_alive():
            thread.join(timeout=30)
        self._uploader_thread = None
        self._uploader_stop = None

    def _maybe_upload_epoch_artifacts(self):
        now = time.time()
        if now - self._last_epoch_upload_at < 8:
            return
        self._last_epoch_upload_at = now
        uploaded = self._upload_pending_artifacts(run_status="training", raise_errors=False)
        if uploaded:
            self._log(f"Uploaded {uploaded} updated artifact(s) for epoch {self.current_epoch or '-'}.")

    # ----------------------------------------------------------- finalize

    def upload(self):
        if not self.api_token:
            raise RuntimeError("Resolve the bundle before uploading results.")
        self._set_status("Uploading results", "Sending checkpoints and logs back to VisoLabel.", "active")
        uploaded = self._upload_pending_artifacts(run_status="completed", raise_errors=True)
        if uploaded == 0:
            self._log("No new uploadable artifacts found.")
        try:
            self._post("/api/v1/colab/finalize-run/", {"token": self.api_token})
        except Exception as exc:
            self._log(f"Finalize endpoint not available: {exc}")
        elapsed = int(time.time() - self.started_at) if self.started_at else 0
        self._set_status("Complete", f"Uploaded {uploaded} new artifact(s) in {elapsed}s.", "ok")

    # ------------------------------------------------------- click handlers

    def _set_busy(self, busy: bool):
        self.run_button.disabled = busy
        self.resolve_button.disabled = busy
        self.train_button.disabled = busy or self.manifest is None
        self.upload_button.disabled = busy or not self.api_token or not self.training_finished

    def _run_action(self, action):
        self._set_busy(True)
        try:
            action()
        except Exception as exc:
            self._set_status("Error", str(exc), "err")
            self._log(f"ERROR: {exc}")
            raise
        finally:
            self._set_busy(False)

    def _on_resolve(self, _):
        self._run_action(self.resolve_bundle)

    def _on_train(self, _):
        self._run_action(self.train)

    def _on_upload(self, _):
        self._run_action(self.upload)

    def _on_run_full(self, _):
        def pipeline():
            self.resolve_bundle()
            self.train()
            self.upload()
        self._run_action(pipeline)


gui = VisoLabelColabGUI()
gui.display()
